In [1]:
import Gmsh: gmsh
using Gridap, GridapGmsh

[ Info: Precompiling Gridap [56d4f2e9-7ea1-5844-9cf6-b9c51ca7ce8e] 
[ Info: Precompiling GridapGmsh [3025c34a-b394-11e9-2a55-3fee550c04c8] (cache misses: wrong dep version loaded (2))


In [2]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square")
    gmsh.model.geo.addPoint(0,0,0,1,1) # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1,0,0,1,2)
    gmsh.model.geo.addPoint(1,1,0,1,3)
    gmsh.model.geo.addPoint(0,1,0,1,4)
    gmsh.model.geo.addLine(1,2,1)
    gmsh.model.geo.addLine(2,3,2) # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3,4,3)
    gmsh.model.geo.addLine(4,1,4)
        
    gmsh.model.geo.addCurveLoop([1,2,3,4],1)
        
    gmsh.model.geo.addPlaneSurface([1],1)

    N = Int(round(1/h))  # number of elements per edge

    gmsh.model.geo.mesh.setTransfiniteCurve(1, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(2, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(3, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(4, N+1)

    #triangulation
    gmsh.model.geo.mesh.setTransfiniteSurface(1)

    gmsh.model.geo.synchronize()

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")
    gmsh.model.mesh.generate(2)

    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end



create_square_model (generic function with 1 method)

In [3]:
h = 0.025; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000507207s, CPU 0.000257s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Transfinite)
Info    : Done meshing 2D (Wall 0.00250962s, CPU 0.000688s)
Info    : 1681 nodes 3364 elements


TrialFESpace()

In [4]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [5]:
n=7
Δt, T = 2.0^(-n), 2*π/3 # time step, final time
num_steps = Int(round(T / Δt))

268

In [6]:
# smooth u(x,t) = x*(1-x) * y(1-y) * sin(3t) 
g(x) = x[1]*(1-x[1]) *x[2]*(1-x[2])
Δg(x) = -2*x[2]*(1 - x[2]) - 2*x[1]*(1 - x[1])

Δg (generic function with 1 method)

In [7]:
#manufactured f(t,x) from our smooth u(x,t).
f(t, x) = (-9*g(x) - Δg(x))*sin(3*t) 
f(t) = x -> f(t,x)

f (generic function with 2 methods)

In [8]:
u_exact(t, x) = g(x) * sin(3t) 
v_exact(t, x) = 3*g(x) * cos(3t)
u₀ = x -> 0.0
v₀ = x -> 0.0

#5 (generic function with 1 method)

In [9]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

LHSpos = (2/Δt)*M + (Δt/2)*K
RHSpos = (2/Δt)*M - (Δt/2)*K

1521×1521 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10337 stored entries:
⎡⢿⣷⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠻⢿⣷⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠻⢿⣷⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠻⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣿⣿⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣵⣿⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣆⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⣿⣿⣦⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⢿⣷⣦⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⢿⣷⣦⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⢿⣷⎦

In [10]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
vh0 = interpolate_everywhere(v₀, U)   # V^0

SingleFieldFEFunction():
 num_cells: 3200
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 8446138602874065642

In [11]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    pvd[0.0] = createvtk(Ω, "tmp/det_wavev3_0.vtu",
                         cellfields=["uh" => uh0, "vh" => vh0, "e" => x -> 0.0])

    # Work on dof-vectors
    uvec = get_free_dof_values(uh0)  # U^0
    vvec = get_free_dof_values(vh0)  # V^0

    e = nothing

    for n in 1:(num_steps-1)
        t_prev = (n-1)*Δt
        t_mid  = (n-0.5)*Δt
        t_now  = n*Δt

        b(v) = ∫( f(t_prev)*v )dΩ
        F = assemble_vector(b, V)


        #Solve for U^n
        rhs_u = RHSpos * uvec + (2*M)*vvec  + 2*F
        uvec_new = LHSpos \ rhs_u
        
        #update V^n
        #rhs_v = M*vvec - (Δt/2)*K*(uvec_new + uvec) + Δt*Fmid
        rhs_v = ((2/Δt) * M) * (uvec_new - uvec) - M*vvec - F
        vvec_new = M \ rhs_v
        


        # Error
        uh = FEFunction(U, uvec_new)
        vh = FEFunction(U, vvec_new)
        uex = x -> u_exact(t_now, x)
        
        e = uex - uh

        # save
        vtufilename = "tmp/det_wavev3_$(n).vtu"
        pvd[t_now] = createvtk(Ω, vtufilename, cellfields=["uh" => uh, "vh" => vh, "e" => e])

        # update
        uvec = uvec_new
        vvec = vvec_new
    end

    el2 = sqrt(sum( ∫(e*e)*dΩ ))
    println("L2 error norm: $el2")
end

L2 error norm: 0.5426883847578906


269-element Vector{String}:
 "wave.pvd"
 "tmp/det_wavev3_0.vtu"
 "tmp/det_wavev3_1.vtu"
 "tmp/det_wavev3_2.vtu"
 "tmp/det_wavev3_3.vtu"
 "tmp/det_wavev3_4.vtu"
 "tmp/det_wavev3_5.vtu"
 "tmp/det_wavev3_6.vtu"
 "tmp/det_wavev3_7.vtu"
 "tmp/det_wavev3_8.vtu"
 "tmp/det_wavev3_9.vtu"
 "tmp/det_wavev3_10.vtu"
 "tmp/det_wavev3_11.vtu"
 ⋮
 "tmp/det_wavev3_256.vtu"
 "tmp/det_wavev3_257.vtu"
 "tmp/det_wavev3_258.vtu"
 "tmp/det_wavev3_259.vtu"
 "tmp/det_wavev3_260.vtu"
 "tmp/det_wavev3_261.vtu"
 "tmp/det_wavev3_262.vtu"
 "tmp/det_wavev3_263.vtu"
 "tmp/det_wavev3_264.vtu"
 "tmp/det_wavev3_265.vtu"
 "tmp/det_wavev3_266.vtu"
 "tmp/det_wavev3_267.vtu"